# Étude de cas VIGA — 1. Entraînement du détecteur de poissons et de points clés (YOLO-pose)

Données : annotations du dataset FHA-MTT (Chen et al., 2026, *Smart Agricultural Technology* 13:101830 ; Zenodo 10.5281/zenodo.17518455, CC BY 4.0), converties au format YOLO-pose par `labelme_vers_yolo_pose.py`.

Modèle de référence : **YOLOv8n-pose**, avec les réglages de l'article (image 640, lot 16, 100 époques, lr0 = 0,01, weight decay = 0,0005), pour comparer directement au tableau 3 de l'article.

Réglages Kaggle : *Accelerator* = GPU (P100 ou T4), *Internet* = activé, **un seul** dataset d'annotations en entrée :
`fish-pose-yolo` (découpage aléatoire d'origine), `fish-pose-yolo-blocs` (découpage par blocs de temps) ou `fish-pose-yolo-alea` (témoin aléatoire de même taille). Les fichiers de sortie portent le nom du découpage.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics

## Préparer le jeu de données
Le dossier d'entrée de Kaggle est en lecture seule : on copie le jeu de données dans `/kaggle/working` et on réécrit le fichier de configuration avec les bons chemins.

In [ ]:
import glob, os, shutil, zipfile, json
from pathlib import Path

RACINE_ENTREE = Path(os.environ.get("RACINE_ENTREE", "/kaggle/input"))
TRAVAIL = Path(os.environ.get("TRAVAIL", "/kaggle/working"))
DATA = TRAVAIL / "fish_pose_yolo"

if DATA.exists():
    shutil.rmtree(DATA)
yamls = glob.glob(str(RACINE_ENTREE / "**" / "fish-pose.yaml"), recursive=True)
zips = glob.glob(str(RACINE_ENTREE / "**" / "fish_pose_yolo*.zip"), recursive=True)
sources = yamls or zips
if len(sources) > 1:
    raise RuntimeError(f"Plusieurs datasets d'annotations en entrée : {sources}. N'en garde qu'un.")
if yamls:
    shutil.copytree(Path(yamls[0]).parent, DATA)
elif zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(DATA)
else:
    raise FileNotFoundError("Ajoute un dataset d'annotations (fish-pose-yolo...) en entrée du notebook (Add Input).")
chemin_source = sources[0].lower() + (Path(yamls[0]).read_text().lower() if yamls else "")
SUFFIXE = "_blocs" if "bloc" in chemin_source else "_alea" if "alea" in chemin_source else ""
print("Découpage :", SUFFIXE.strip("_") or "aléatoire d'origine")

for part in ["train", "val", "test"]:
    n = len(list((DATA / "images" / part).glob("*.jpg")))
    print(f"{part:5s} : {n} images")

# Listes d'images de test séparées : images complètes du bassin vs recadrages sur un poisson
import cv2
completes, recadrages = [], []
for p in sorted((DATA / "images" / "test").glob("*.jpg")):
    w = cv2.imread(str(p)).shape[1]
    (completes if w >= 600 else recadrages).append(str(p))
(DATA / "test_completes.txt").write_text("\n".join(completes) + "\n")
(DATA / "test_recadrages.txt").write_text("\n".join(recadrages) + "\n")
print(f"test : {len(completes)} images completes, {len(recadrages)} recadrages")

def ecrire_yaml(nom, test):
    chemin = DATA / nom
    chemin.write_text(
        f"path: {DATA}\ntrain: images/train\nval: images/val\ntest: {test}\n"
        "kpt_shape: [3, 3]\nflip_idx: [0, 1, 2]\nnames:\n  0: fish\n"
        "kpt_names:\n  0: [head, body, fin]\n"
    )
    return str(chemin)

YAML = ecrire_yaml("fish-pose.yaml", "images/test")
YAML_COMPLETES = ecrire_yaml("fish-pose-test-completes.yaml", "test_completes.txt")
YAML_RECADRAGES = ecrire_yaml("fish-pose-test-recadrages.yaml", "test_recadrages.txt")

## Entraînement
Réglages de l'article (SGD, lr0 = 0,01). `seed` fixée pour la reproductibilité. Compte environ 1 h sur GPU.

In [ ]:
from ultralytics import YOLO
import torch

EPOQUES = int(os.environ.get("EPOQUES", 100))
model = YOLO("yolov8n-pose.pt")
resultats = model.train(
    data=YAML, imgsz=640, epochs=EPOQUES, batch=16,
    optimizer="SGD", lr0=0.01, weight_decay=0.0005, patience=30, seed=42, deterministic=True,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(TRAVAIL / "runs"), name="yolov8n_pose" + SUFFIXE, exist_ok=True, plots=True,
)
DOSSIER_RUN = Path(resultats.save_dir)
print("Résultats dans", DOSSIER_RUN)

## Évaluation sur le jeu de test
Trois évaluations : tout le test, puis séparément les images complètes du bassin (le cas réel du suivi) et les recadrages.

In [ ]:
import pandas as pd

meilleur = YOLO(str(DOSSIER_RUN / "weights" / "best.pt"))
lignes = []
evaluations = [("test (tout)", YAML, 1), ("test images completes", YAML_COMPLETES, len(completes)),
               ("test recadrages", YAML_RECADRAGES, len(recadrages))]
for nom, y, n in evaluations:
    if n == 0:
        continue
    m = meilleur.val(data=y, split="test", imgsz=640, batch=16, plots=False, verbose=False,
                     project=str(TRAVAIL / "runs"), name="val_" + nom.split()[-1], exist_ok=True)
    lignes.append({
        "jeu": nom + SUFFIXE.replace("_", " (") + (")" if SUFFIXE else ""),
        "precision_boite": m.box.mp, "rappel_boite": m.box.mr,
        "mAP50_boite": m.box.map50, "mAP50-95_boite": m.box.map,
        "precision_pose": m.pose.mp, "rappel_pose": m.pose.mr,
        "mAP50_pose": m.pose.map50, "mAP50-95_pose": m.pose.map,
    })
tableau = pd.DataFrame(lignes)
reference = pd.DataFrame([{
    "jeu": "Article, YOLOv8n-pose (tableau 3)",
    "precision_boite": 0.9681, "rappel_boite": 0.9713, "mAP50_boite": 0.9916, "mAP50-95_boite": 0.6440,
    "precision_pose": 0.9692, "rappel_pose": 0.9732, "mAP50_pose": 0.9922, "mAP50-95_pose": 0.9769,
}])
tableau = pd.concat([tableau, reference], ignore_index=True)
tableau.to_csv(TRAVAIL / f"metriques_detection{SUFFIXE}.csv", index=False)
tableau.round(3)

## Exemples de prédictions et sauvegarde des poids

In [ ]:
exemples = sorted((DATA / "images" / "test").glob("*.jpg"), key=lambda p: -p.stat().st_size)[:4]
sortie = meilleur.predict([str(p) for p in exemples], imgsz=640, conf=0.25, save=True,
                          project=str(TRAVAIL / "runs"), name="exemples", exist_ok=True, verbose=False)
shutil.copy(DOSSIER_RUN / "weights" / "best.pt", TRAVAIL / f"best_yolov8n_pose{SUFFIXE}.pt")
print("Poids :", TRAVAIL / f"best_yolov8n_pose{SUFFIXE}.pt")
print("Courbes :", DOSSIER_RUN / "results.png")
print("Exemples :", TRAVAIL / "runs" / "exemples")

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(DOSSIER_RUN / "results.png"), width=900))
for p in sorted((TRAVAIL / "runs" / "exemples").glob("*.jpg"))[:2]:
    display(Image(filename=str(p), width=900))